In [1]:
# PHASE 2 - ABLATION 7: CROSS-DATASET TRANSFER of the frozen features (Part-I "Study B" protocol)
# Camelyon17-WILDS tumour probe is fitted ONCE (logistic regression, class_weight='balanced') and then applied, never refit, to PatchCamelyon, NCT-CRC-HE-100K and LC25000.
# Models: ORIGINAL | SimCLR control | self | ADJACENCY | CCSM ms / ms+Fourier (ResNet18 also ms-random) (+ Lunit combined) for resnet18, vit_small, swin_tiny, kaiko, lunit, phikon; seeds 42/43/44.  ResNet18 / ViT-S / Swin-T adjacency+self = peesarisathvikreddy/adjclv2-checkpoints (v2 recipe), Kaiko / Lunit / Phikon = sethu2954/adjc-* (v4 recipe).
# Kaggle: GPU on, Internet on.  Checkpoints are found/downloaded automatically (sethu2954/adjc-*-checkpoints, badrinathachanta/ccsm-checkpoints-1 and -2, peesarisathvikreddy/adjclv2-checkpoints);
# the combined Lunit checkpoints (abl5-comb_vit_small_neighbor_s<seed>_best.pt) are in the dataset badrinathachanta/ablation-combined-checkpoints -> ATTACH it as an input (skipped if absent).
# Estimated runtime ~5-6 h (102 models) (data decoding ~30 min once).  Resumable (a small .json per model); stops before 11 h.
!pip install -q timm datasets kagglehub transformers


import os, gc, json, glob, random, time, shutil, warnings, pickle
from collections import defaultdict
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')
T_START = time.time()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUR_SIZE = [224]          # network input size of the model currently evaluated (ResNet18 = 96, the others 224)

def load_camelyon():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset("wltjr1007/Camelyon17-WILDS"); parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ Camelyon17-WILDS: {len(full):,} patches"); return full

def patient_split_by_centre(meta, val_frac, seed, centres=(0, 1, 2, 3, 4)):
    """Per-centre, patient-level split (identical convention to the Part-I probe scripts)."""
    tr, te = {}, {}
    for c in centres:
        pats = sorted(meta.loc[meta['center'] == c, 'patient'].unique().tolist())
        random.Random(seed + c).shuffle(pats)
        if len(pats) < 2: tr[c], te[c] = set(pats), set(); continue
        n_te = min(max(1, int(round(val_frac * len(pats)))), len(pats) - 1)
        te[c], tr[c] = set(pats[:n_te]), set(pats[n_te:])
    return tr, te

def stratified_sample(labels, n_per_class, seed):
    rng = np.random.RandomState(seed)
    pos, neg = np.where(labels == 1)[0], np.where(labels == 0)[0]
    rng.shuffle(pos); rng.shuffle(neg)
    k = min(n_per_class, len(pos), len(neg))
    if k < n_per_class: print(f"   ⚠ only {len(pos)} pos / {len(neg)} neg available -> {k}/class")
    idx = np.concatenate([pos[:k], neg[:k]]); rng.shuffle(idx); return idx

@torch.no_grad()
def embed(enc, X_u8, bs=128, amp=False, size=None):
    """uint8 [N,3,h,w] -> pooled features [N,D] (bilinear up-sampling to `size` on the GPU, ImageNet normalisation)."""
    out = []; size = size or CUR_SIZE[0]
    for i in range(0, len(X_u8), bs):
        x = X_u8[i:i + bs].to(DEVICE).float() / 255
        if x.shape[-1] != size: x = F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False, antialias=True)
        x = (x - MEAN.to(DEVICE)) / STD.to(DEVICE)
        with torch.autocast('cuda', enabled=amp and DEVICE.type == 'cuda'):
            out.append(enc.encode(x).float().cpu())
    return torch.cat(out).numpy()

def load_uint8(ds, indices, size, desc):
    """Decode `indices` of an HF dataset to a uint8 tensor [n,3,size,size] (PIL bilinear resize if needed)."""
    from tqdm.auto import tqdm
    from torchvision import transforms
    tf = transforms.Resize((size, size))
    out = np.zeros((len(indices), size, size, 3), np.uint8)
    for k, i in enumerate(tqdm(indices, desc=desc)):
        im = ds[int(i)]['image']
        if im.mode != 'RGB': im = im.convert('RGB')
        if im.size != (size, size): im = tf(im)
        out[k] = np.asarray(im)
    return torch.from_numpy(out).permute(0, 3, 1, 2).contiguous()


# ---- frozen encoders / checkpoint loading (same code as the D7/D8 evaluations) ----
import os, gc, json, glob, random, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')


class EC:
    RUN_FAMILIES = ['kaiko', 'lunit', 'phikon']
    SEEDS = [42, 43, 44]
    BUDGET_H = 11.0                       # stop starting new chunks when the next one would not fit
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CKPT_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASETS = ["sethu2954/adjc-kaiko-checkpoints", "sethu2954/adjc-lunit-checkpoints", "sethu2954/adjc-phikon-checkpoints"]   # fallback if not attached
    PRIOR_RESULT_ROOTS = ["/kaggle/input"]                                  # earlier runs' outputs (p2_eval_all/*.json) are reused
    FEATURE_DIR = "/kaggle/working/unused_features"
    RESULTS_DIR = "/kaggle/working/unused_results"
    KEEP_FEATURES = False

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    FAMILIES = {   # kind, hub id / timm name, input size, fp16 extraction, Part-I LOCO of the original (pipeline check; None = ImageNet control)
        'resnet18':  dict(kind='resnet', hub=None, size=96,  amp=False, ref=0.9528, label='ImageNet ResNet18'),
        'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, amp=False, ref=None, label='ImageNet ViT-S/16'),
        'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, amp=False, ref=None, label='ImageNet Swin-T'),
        'kaiko':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, amp=False, ref=0.9873, label='Kaiko ViT-S/16'),
        'lunit':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, amp=False, ref=0.9814, label='Lunit ViT-S/16'),
        'phikon':    dict(kind='hf', hub='owkin/phikon', size=224, amp=True, ref=0.9818, label='Phikon ViT-B/16'),
    }
    ARMS = {}
    DEFAULT_ARMS = ['neighbor', 'self']
    BACKBONE = {'kaiko': 'vit_small', 'lunit': 'vit_small', 'phikon': 'phikon'}      # name used in the checkpoint file names
    PART1_OTHERS = "Part-I LOCO references: Stain Aug 0.9884 | Kaiko 0.9873 | Phikon 0.9818 | Lunit 0.9814 | ImageNet ResNet18 0.9528"

    @staticmethod
    def arms(fam):
        return EC.ARMS.get(fam, EC.DEFAULT_ARMS)

    @staticmethod
    def chunks(fam):
        """[(chunk_models)] : chunk 1 = original + seed 42, then one chunk per further seed.  Model name = '<family>__ctrl' / '<family>__<arm>_s<seed>'."""
        out = []
        for k, s in enumerate(EC.SEEDS):
            c = [f'{fam}__ctrl'] if k == 0 else []
            c += [f'{fam}__{a}_s{s}' for a in EC.arms(fam)]
            out.append(c)
        return out


def parse(name):
    fam, rest = name.split('__')
    if rest == 'ctrl':
        return fam, 'ctrl', None
    arm, s = rest.rsplit('_s', 1)
    return fam, arm, int(s)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders (same construction as Part I / the training notebooks)
class Encoder(nn.Module):
    def __init__(self, fam):
        super().__init__()
        spec = EC.FAMILIES[fam]; self.pool = None
        if spec['kind'] == 'resnet':
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(net.children())[:-1]); self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif spec['kind'] == 'hf':
            from transformers import AutoModel
            class PhikonBackbone(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.vit = AutoModel.from_pretrained(spec['hub'])
                def forward(self, x):
                    return self.vit(pixel_values=x).last_hidden_state[:, 0]           # CLS token, as in Part I
            self.backbone = PhikonBackbone(); self.dim = self.backbone.vit.config.hidden_size
        else:
            import timm
            self.backbone = timm.create_model(spec['hub'], pretrained=True, num_classes=0)
            self.dim = self.backbone.num_features

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    if glob.glob("/kaggle/input/**/adjclv4-*_best.pt", recursive=True):
        return
    import kagglehub
    for slug in EC.KAGGLE_CKPT_DATASETS:
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            EC.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = f"adjclv4-{fam}_{EC.BACKBONE[fam]}_{arm}_s{seed}_best.pt"
    for root in EC.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    fam, arm, seed = parse(name)
    if arm == 'ctrl':
        return Encoder(fam)
    path = find_checkpoint(fam, arm, seed)
    if path is None:
        return None
    enc = Encoder(fam)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad = [k for k in unexpected if not k.startswith(('head.', 'adv_head.'))]
    assert not missing and not bad, f"{name}: missing={missing[:3]} unexpected={bad[:3]}"
    print(f"  [{name}] loaded {os.path.basename(path)}  (epoch {ck.get('epoch','?')}, final loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{EC.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    fam = parse(todo[0])[0]; size = EC.FAMILIES[fam]['size']; amp = EC.FAMILIES[fam]['amp'] and device.type == 'cuda'
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, {size}), batch_size=EC.EXTRACT_BATCH, shuffle=False,
                        num_workers=EC.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    for batch in tqdm(loader, desc=f"extract {fam}: {len(mods)} models"):
        x = batch[size].to(device, non_blocking=True); B = x.size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(x)
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(EC.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, C=EC.LR_C, class_weight='balanced',
                             random_state=EC.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], EC.BOOT_N, EC.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(EC.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in EC.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(EC.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(EC.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], EC.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], EC.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))




# ====================================================================
# MODELS: original + {SimCLR control, self, adjacency, CCSM ms / fms, combined}  x seeds  (Kaiko / Lunit / Phikon)
# ====================================================================
FAM_ARMS = {'resnet18':  ['control', 'self', 'neighbor', 'ms', 'msrand', 'fms'],
            'vit_small': ['control', 'self', 'neighbor', 'ms', 'fms'],
            'swin_tiny': ['control', 'self', 'neighbor', 'ms', 'fms'],
            'kaiko':  ['control', 'self', 'neighbor', 'ms', 'fms'],
            'lunit':  ['control', 'self', 'neighbor', 'ms', 'fms', 'comb'],
            'phikon': ['control', 'self', 'neighbor', 'ms', 'fms']}
ARM_LABEL = {'ctrl': 'ORIGINAL pretrained weights', 'msrand': 'CCSM: MixStyle, random partners', 'control': 'SimCLR control (CCSM recipe)', 'self': 'self control (adjacency recipe)', 'neighbor': 'ADJACENCY (Method 1)',
             'ms': 'CCSM: MixStyle', 'fms': 'CCSM: MixStyle + Fourier', 'comb': 'COMBINED adjacency + CCSM'}
ARM_ORDER = ['ctrl', 'control', 'self', 'neighbor', 'ms', 'msrand', 'fms', 'comb']
ADJ_SLUGS = {'resnet18': "peesarisathvikreddy/adjclv2-checkpoints", 'vit_small': "peesarisathvikreddy/adjclv2-checkpoints", 'swin_tiny': "peesarisathvikreddy/adjclv2-checkpoints",
             'kaiko': "sethu2954/adjc-kaiko-checkpoints", 'lunit': "sethu2954/adjc-lunit-checkpoints", 'phikon': "sethu2954/adjc-phikon-checkpoints"}
COMB_SLUG = "badrinathachanta/ablation-combined-checkpoints"      # Lunit combined (adjacency + CCSM) checkpoints from ABL5
CCSM_SLUGS = {'resnet18': "badrinathachanta/ccsm-checkpoints-1", 'vit_small': "badrinathachanta/ccsm-checkpoints-1", 'swin_tiny': "badrinathachanta/ccsm-checkpoints-1",
              'kaiko': "badrinathachanta/ccsm-checkpoints-2", 'lunit': "badrinathachanta/ccsm-checkpoints-2", 'phikon': "badrinathachanta/ccsm-checkpoints-2"}

def ckpt_fname(fam, arm, seed):
    if arm in ('self', 'neighbor') and fam in ('resnet18', 'vit_small', 'swin_tiny'): return f"adjclv2_{fam}_{arm}_s{seed}_best.pt"      # adjacency v2 recipe (ResNet18 / ViT-S / Swin-T)
    if arm in ('self', 'neighbor'): return f"adjclv4-{fam}_{'phikon' if fam == 'phikon' else 'vit_small'}_{arm}_s{seed}_best.pt"
    if arm in ('control', 'ms', 'msrand', 'fms'): return f"ccsm_{fam}_{arm}_s{seed}_best.pt"
    if arm == 'comb': return f"abl5-comb_vit_small_neighbor_s{seed}_best.pt"
    raise ValueError(arm)

def ensure_checkpoint_roots():
    import kagglehub
    for fam in RUN_FAMILIES:
        pairs = [('ccsm', CCSM_SLUGS[fam])] + ([('adjacency', ADJ_SLUGS[fam])] if fam in ADJ_SLUGS else []) + ([('comb', COMB_SLUG)] if fam == 'lunit' else [])
        for kind, slug in pairs:
            probe = ckpt_fname(fam, {'adjacency': 'neighbor', 'comb': 'comb'}.get(kind, 'ms'), SEEDS[0])
            if glob.glob(os.path.join('/kaggle/input', '**', probe), recursive=True): continue
            try:
                path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", slug); EC.CKPT_SEARCH_ROOTS.append(path)
            except Exception as e:
                print(f"⚠ could not download {slug}: {e}")

def find_checkpoint(fam, arm, seed):
    fname = ckpt_fname(fam, arm, seed)
    for root in ['/kaggle/input'] + list(EC.CKPT_SEARCH_ROOTS):
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits: return hits[0]
    return None

def model_names():
    out = []
    for fam in RUN_FAMILIES:
        out.append(f'{fam}__ctrl')
        for s in SEEDS: out += [f'{fam}__{a}_s{s}' for a in FAM_ARMS[fam]]
    return out


def have_result(name): return os.path.exists(f"{RES_DIR}/{name}.json")
def prior_results():
    n = 0
    for p in glob.glob(f'/kaggle/input/**/{os.path.basename(RES_DIR)}/*.json', recursive=True):
        dst = os.path.join(RES_DIR, os.path.basename(p))
        if not os.path.exists(dst): shutil.copy2(p, dst); n += 1
    if n: print(f"✓ reused {n} finished results from an earlier run")

def _ms(v):
    v = np.array(v, dtype=float); return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))

def run_all(process_fn, device):
    """process_fn(name, encoder) -> dict (JSON-able).  Resumable; stops cleanly before BUDGET_H."""
    T0 = time.time(); last = 0.0
    os.makedirs(RES_DIR, exist_ok=True); prior_results(); ensure_checkpoint_roots()
    names = model_names(); todo = [n for n in names if not have_result(n)]
    print(f"{len(names) - len(todo)}/{len(names)} models already done")
    for n in todo:
        el = (time.time() - T0) / 3600
        if el + max(last, 0.15) > BUDGET_H:
            print(f"\n⏸ time budget reached ({el:.1f} h). Save this version and re-run with this OUTPUT attached as an input to finish the rest."); break
        t0 = time.time()
        enc = build_model(n)
        if enc is None:
            print(f"  [{n}] checkpoint not found -> skipped"); continue
        enc = enc.to(device).eval(); CUR_SIZE[0] = EC.FAMILIES[n.split('__')[0]]['size']
        res = process_fn(n, enc); res['model'] = n
        json.dump(res, open(f"{RES_DIR}/{n}.json", 'w'))
        del enc; gc.collect(); torch.cuda.empty_cache()
        last = (time.time() - t0) / 3600


"""
Protocol (identical to the Part-I cross_dataset_transfer script): probe-train = 3000 patches/class from the TRAIN patients of all five Camelyon17 centres (per-centre patient split,
30% test patients, seed 42); in-domain test = 1000/class from the held-out patients; targets: PatchCamelyon test (3000/class), NCT-CRC-HE-100K CRC-VAL-HE-7K split (TUM = positive,
800/class), LC25000 (adenocarcinoma + squamous = malignant, 2000/class).  Probe: LogisticRegression(max_iter=1000, class_weight='balanced') on raw features.  Reported: balanced accuracy and AUROC.
Note: the probe-train patches include centre 2 training patients (as in the Part-I script, whose text describes centres {0,1,3,4}); in-domain = held-out patients, not held-out centres.
"""
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score

RUN_FAMILIES = ['resnet18', 'vit_small', 'swin_tiny', 'kaiko', 'lunit', 'phikon']
SEEDS = [42, 43, 44]
BUDGET_H = 11.0
RES_DIR = "/kaggle/working/abl_transfer_"
DATA_CACHE = "/kaggle/working/transfer_data.pt"
SEED = 42
TARGETS = [dict(name="PatchCamelyon", hf_id="1aurent/PatchCamelyon", split="test", kw=None, n_per_class=3000),
           dict(name="NCT-CRC-HE-100K", hf_id="1aurent/NCT-CRC-HE", split="CRC_VAL_HE_7K", kw=["tum"], n_per_class=800),
           dict(name="LC25000", hf_id="1aurent/LC25000", split="train", kw=["adenocarcinom", "squamous"], n_per_class=2000)]

def binary_labels(ds, kw, name):
    feat = ds.features['label']; names = getattr(feat, 'names', None)
    raw = np.asarray(ds['label'])
    if kw is None or names is None:
        lab = raw.astype(np.int64); print(f"   [{name}] binary label column, positive rate {lab.mean():.3f}"); return lab
    matched = [n for n in names if any(k.lower() in n.lower() for k in kw)]
    print(f"   [{name}] classes {names}  ->  POSITIVE: {matched}")
    if not matched: raise ValueError(f"no class of {names} matched {kw}")
    m = np.array([1 if n in matched else 0 for n in names], dtype=np.int64)
    lab = m[raw.astype(np.int64)]; print(f"   [{name}] positive rate {lab.mean():.3f}"); return lab

def load_target(spec):
    from datasets import load_dataset
    try:
        return load_dataset(spec['hf_id'], split=spec['split'])
    except Exception as e:
        print(f"   (split-only load failed: {str(e)[:100]} -> loading all splits)")
        return load_dataset(spec['hf_id'])[spec['split']]

def prepare_data():
    cached = [DATA_CACHE] + glob.glob('/kaggle/input/**/transfer_data.pt', recursive=True)
    for p in cached:
        if os.path.exists(p):
            print("✓ loading cached data", p); return torch.load(p)
    cam = load_camelyon()
    meta = cam.select_columns(['patient', 'center', 'label']).to_pandas()
    tr_p, te_p = patient_split_by_centre(meta, 0.3, SEED)
    pats = meta['patient'].to_numpy(); cen = meta['center'].to_numpy(); y = meta['label'].to_numpy().astype(np.int64)
    trm = np.zeros(len(meta), bool); tem = np.zeros(len(meta), bool)
    for c in range(5):
        cm = cen == c
        trm |= cm & np.isin(pats, list(tr_p.get(c, set()))); tem |= cm & np.isin(pats, list(te_p.get(c, set())))
    tr_pool, te_pool = np.where(trm)[0], np.where(tem)[0]
    train_sel = tr_pool[stratified_sample(y[tr_pool], 3000, SEED)]
    test_sel = te_pool[stratified_sample(y[te_pool], 1000, SEED + 1)]
    data = {'cam_train': (load_uint8(cam, train_sel, 224, 'camelyon train'), y[train_sel]),
            'cam_test': (load_uint8(cam, test_sel, 224, 'camelyon held-out patients'), y[test_sel])}
    for i, spec in enumerate(TARGETS):
        try:
            ds = load_target(spec); lab = binary_labels(ds, spec['kw'], spec['name'])
            sel = stratified_sample(lab, spec['n_per_class'], SEED + 2 + i)
            data[spec['name']] = (load_uint8(ds, sel, 224, spec['name']), lab[sel])
        except Exception as e:
            print(f"⚠ target {spec['name']} could not be prepared: {e}")
    torch.save(data, DATA_CACHE); print("✓ data cached to", DATA_CACHE)
    return data

def metrics(clf, X, y):
    pred = clf.predict(X); proba = clf.predict_proba(X)[:, 1]
    return {'bal_acc': 100 * balanced_accuracy_score(y, pred), 'acc': 100 * accuracy_score(y, pred), 'auroc': float(roc_auc_score(y, proba))}

DATA = {}
def process(name, enc):
    fam = name.split('__')[0]; amp = EC.FAMILIES[fam]['amp']; t0 = time.time()
    Xtr = embed(enc, DATA['cam_train'][0], amp=amp); ytr = DATA['cam_train'][1]
    clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr, ytr)
    res = {'indomain': metrics(clf, embed(enc, DATA['cam_test'][0], amp=amp), DATA['cam_test'][1]), 'targets': {}}
    for spec in TARGETS:
        if spec['name'] in DATA:
            res['targets'][spec['name']] = metrics(clf, embed(enc, DATA[spec['name']][0], amp=amp), DATA[spec['name']][1])
    res['mean_drop'] = float(np.mean([res['indomain']['bal_acc'] - t['bal_acc'] for t in res['targets'].values()])) if res['targets'] else float('nan')
    print(f"  [{name}] in-domain {res['indomain']['bal_acc']:.1f}% | " + " | ".join(f"{k} {v['bal_acc']:.1f}%" for k, v in res['targets'].items()) + f" | mean drop {res['mean_drop']:.1f} | {(time.time()-t0)/60:.1f} min")
    return res

def summarize():
    res = {}
    for p in glob.glob(f"{RES_DIR}/*.json"):
        r = json.load(open(p)); res[r['model']] = r
    tnames = [t['name'] for t in TARGETS]
    print("\n" + "=" * 128 + "\nCROSS-DATASET TRANSFER  (balanced accuracy %, probe fitted on Camelyon17-WILDS only; mean ± std over seeds)\n" + "=" * 128)
    rows = []
    for fam in RUN_FAMILIES:
        print(f"\n--- {fam} ---\n  {'':34s}{'in-domain':>14s}" + "".join(f"{t[:14]:>16s}" for t in tnames) + f"{'mean drop':>14s}")
        per = {a: {} for a in ARM_ORDER}
        for n, r in res.items():
            f, a, s = parse(n)
            if f == fam: per[a][s] = r
        for a in ARM_ORDER:
            d = per[a]
            if not d: continue
            vals = list(d.values())
            def cell(get):
                m, sd = _ms([get(v) for v in vals]); return f"{m:6.1f}" + (f"±{sd:4.1f}" if len(vals) > 1 else "     ")
            line = f"  {ARM_LABEL[a]:34s}{cell(lambda v: v['indomain']['bal_acc']):>14s}"
            row = dict(family=fam, method=ARM_LABEL[a], seeds=len(vals), indomain=cell(lambda v: v['indomain']['bal_acc']).strip())
            for t in tnames:
                if all(t in v['targets'] for v in vals):
                    c = cell(lambda v: v['targets'][t]['bal_acc']); line += f"{c:>16s}"; row[t] = c.strip()
                else: line += f"{'--':>16s}"
            c = cell(lambda v: v['mean_drop']); line += f"{c:>14s}"; row['mean_drop'] = c.strip()
            print(line); rows.append(row)
        def paired(a, b, label):
            if not (per.get(a) and per.get(b)): return
            common = sorted(set(per[a]) & set(per[b]))
            if not common: return
            out = []
            for t in tnames:
                g = [per[a][s]['targets'][t]['bal_acc'] - per[b][s]['targets'][t]['bal_acc'] for s in common if t in per[a][s]['targets'] and t in per[b][s]['targets']]
                if g: out.append(f"{t[:10]} {np.mean(g):+5.1f} ({sum(x > 0 for x in g)}/{len(g)})")
            g = [per[a][s]['mean_drop'] - per[b][s]['mean_drop'] for s in common]
            print(f"      {label:38s} " + " | ".join(out) + f" | mean-drop change {np.mean(g):+5.1f} (negative = smaller drop)")
        for a, b, lab in [('neighbor', 'self', 'adjacency minus self'), ('neighbor', 'ctrl', 'adjacency minus ORIGINAL'), ('fms', 'control', 'CCSM(MS+F) minus SimCLR control'),
                          ('fms', 'ctrl', 'CCSM(MS+F) minus ORIGINAL'), ('comb', 'neighbor', 'combined minus adjacency')]:
            if a in per and b in per: paired(a, b, lab)
    pd.DataFrame(rows).to_csv(f"{RES_DIR}/transfer_summary.csv", index=False)
    print("\nNote: ResNet18 runs at 96 px (images down-sampled from the 224 px cache); the others at 224 px.\nPart-I reference (Table V, balanced acc %): Kaiko 87.7 | 75.2 / 63.3 / 60.7 ; Lunit 85.9 | 77.4 / 83.9 / 63.4 ; Phikon 87.0 | 82.2 / 51.2 / 91.8 (in-domain | PCam / NCT / LC25000)")
    print("Caveat: adjacency / CCSM models saw the UNLABELED Camelyon17 images of centres 0,1,3,4 during pretraining; the three external datasets were never seen by any model.")

if True:
    DATA.update(prepare_data())
    run_all(process, DEVICE)
    summarize()

README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

  [phikon__neighbor_s44] loaded adjclv4-phikon_phikon_neighbor_s44_best.pt  (epoch 24, final loss 0.6028)
  [phikon__neighbor_s44] in-domain 90.4% | PatchCamelyon 83.5% | NCT-CRC-HE-100K 72.2% | LC25000 52.0% | mean drop 21.1 | 1.0 min


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  [phikon__ms_s44] loaded ccsm_phikon_ms_s44_best.pt  (epoch 6, final loss 0.0467)
  [phikon__ms_s44] in-domain 78.9% | PatchCamelyon 84.4% | NCT-CRC-HE-100K 46.9% | LC25000 74.8% | mean drop 10.2 | 1.0 min


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  [phikon__fms_s44] loaded ccsm_phikon_fms_s44_best.pt  (epoch 6, final loss 0.0713)
  [phikon__fms_s44] in-domain 89.1% | PatchCamelyon 85.2% | NCT-CRC-HE-100K 62.0% | LC25000 69.0% | mean drop 17.1 | 1.0 min

CROSS-DATASET TRANSFER  (balanced accuracy %, probe fitted on Camelyon17-WILDS only; mean ± std over seeds)

--- resnet18 ---
                                         in-domain   PatchCamelyon  NCT-CRC-HE-100         LC25000     mean drop
  ORIGINAL pretrained weights            77.1            74.7            68.2            61.4           9.0     
  SimCLR control (CCSM recipe)           73.5± 5.0       73.4± 0.5       71.2± 8.2       45.1± 1.4     10.3± 6.2
  self control (adjacency recipe)        89.8± 0.4       75.3± 0.2       57.9± 1.6       70.0± 2.5     22.0± 1.2
  ADJACENCY (Method 1)                   91.5± 0.6       78.4± 1.2       57.5± 2.1       71.0± 3.2     22.5± 1.8
  CCSM: MixStyle                         70.7± 7.4       68.9± 0.8       75.3± 4.6       55.8± 3.5